# Overall any lab completed and any vital completed

File is mostly commented out because it is dependent on structure of private, health data

In [ ]:
'''
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
'''

In [ ]:
'''
#Read in demographics, history, and ses variables
demographics = pd.read_csv('demographics.csv')
off = pd.read_csv('office_visits.csv')
ses = pd.read_csv('ses.csv')

#merge into one dataframe
demographics = demographics.merge(off, right_on = 'preg_id', left_on = 'preg_id', how='left')
demographics = demographics.merge(ses, right_on = 'preg_id', left_on = 'preg_id', how='left')
'''

In [ ]:
'''
his = pd.read_csv('history.csv')
df = pd.merge(demographics, his, how = 'left', left_index = True, right_index = True)
'''

In [ ]:
'''
vitals1 = pd.read_csv('vitals1.csv')
vitals2 = pd.read_csv('vitals2.csv')
vitals = pd.concat([vitals1, vitals2])

labs1 = pd.read_csv('labs1.csv')
labs2 = pd.read_csv('labs2.csv')
labs = pd.concat([labs1, labs2])
'''

In [ ]:
'''
### Lab preprocessing ### 

#making months before last menstural period negative time
labs['mos_priorlmp'] = -1.0 * labs['mos_priorlmp']

#making one time column instead of prior/after
labs['time'] = labs['mos_afterlmp'].fillna(labs['mos_priorlmp'])

#Converting to weeks
labs['time'] = labs['time']*30.4375/7

#only taking labs two years before and up to cutoff
labs['time'] = np.round(labs['time']).astype(int)  
labs = labs[(labs['time'] >= -104) & (labs['time'] <= 36)]

labs = labs.drop(['mos_priorlmp', 'mos_afterlmp'], axis=1)
'''

In [ ]:
'''
labs = labs.set_index('PREG_ID')
'''

In [ ]:
'''
vitals['mos_priorlmp'] = -1.0 * vitals['mos_priorlmp']
vitals['time'] = vitals['mos_afterlmp'].fillna(vitals['mos_priorlmp'])

vitals['time'] = vitals['time']*30.4375/7

vitals['time'] = np.round(vitals['time']).astype(int) 
vitals = vitals[(vitals['time'] >= -104) & (vitals['time'] <= 36)]
vitals = vitals.drop(['mos_priorlmp', 'mos_afterlmp'], axis=1)

#Separate out HR, systolic, and diastolic into separate rows. 
vitals = pd.melt(vitals, id_vars=['preg_id', 'time'], var_name='TEST_TYPE', value_name='KPNC_RESULTN')
'''

In [ ]:
'''
vitals.rename(columns={'preg_id': 'PREG_ID'}, inplace=True)
vitals = vitals.set_index('PREG_ID')
'''

In [ ]:
'''
demographics = df[['mrace', 'parity_c', 'mage']].copy()
demographics['parity'] = (demographics['parity_c'] == '1. 0') * 1
'''

In [ ]:
'''
print(labs.head())
print(vitals.head())
'''

In [ ]:
'''
labs2 = labs[['time', 'TEST_TYPE', 'KPNC_RESULTN']].copy()

tot = pd.concat([vitals, labs2])
print(tot.head())
'''

In [ ]:
'''
print(labs.shape)
print(vitals.shape)
print(tot.shape)
'''

In [ ]:
'''
print(tot.index.nunique())
'''

In [ ]:
'''
cohort = tot.merge(demographics, how='inner', left_index=True, right_index=True)
print(cohort.index.nunique())
'''

In [ ]:
'''
cols = tot['TEST_TYPE'].unique().tolist()
print(cols)
'''

In [ ]:
'''
cols = tot['TEST_TYPE'].unique().tolist()

labs = ['HGB', 'RBC', 'WBC', 'ANC', 'GLU_F', 'GLU_RAN', 'GTTPM_1', 'GTT_1', 'GTT_2', 'GTT_3', 'AST', 'ALT', 'BUN', 'CHLORIDE', 'CREATININE', 'CALCIUM', 'GTT75_2', 'WBC_COR', 'GTTPM_2']
vitals = ['mode_hr', 'avgbp_systolic', 'avgbp_diastolic']

cohort = cohort.dropna(subset=['KPNC_RESULTN'])

df_vitals = cohort[cohort['TEST_TYPE'].isin(vitals)]
df_labs = cohort[cohort['TEST_TYPE'].isin(labs)]

time_range = range(24, 37)

seen_patients_v = set()
cumulative_counts_v = []

seen_patients_l = set()
cumulative_counts_l = []

for t in time_range:
    current_patients_v = set(df_vitals[df_vitals['time'] == t].index)
    seen_patients_v.update(current_patients_v)
    cumulative_counts_v.append(len(seen_patients_v))
    
    current_patients_l = set(df_labs[df_labs['time'] == t].index)
    seen_patients_l.update(current_patients_l)
    cumulative_counts_l.append(len(seen_patients_l))
   
total_patients = cohort.index.nunique()

patient_percentages_v = (np.array(cumulative_counts_v) / total_patients) * 100
patient_percentages_l = (np.array(cumulative_counts_l) / total_patients) * 100

plt.figure(figsize=(10, 10))
plt.plot(time_range, patient_percentages_v, label='Vitals', color='blue', linewidth=3)
plt.plot(time_range, patient_percentages_l, label='Labs', color='red', linewidth=3)

print(patient_percentages_v[-1])
print(patient_percentages_l[-1])

plt.axhline(y=88.71, color='b', linestyle='--', alpha=0.5, linewidth=3)
plt.axhline(y=58.12, color='r', linestyle='--', alpha=0.5, linewidth=3)

plt.axhline(y=96.58, color='b', linestyle='--', alpha=0.5, linewidth=3)
plt.axhline(y=95.03, color='r', linestyle='--', alpha=0.5, linewidth=3)

#plt.xlabel('Time (weeks)', fontsize=24)
plt.ylabel('Percent of Pregnancies', fontsize=24)
plt.title('Late Pregnancy', fontsize=24)
plt.tick_params(axis='both', which='major', labelsize=24)
plt.ylim(-1,100)
#plt.legend(fontsize=20)
plt.grid(True)
plt.show()
'''